# 05 Channel Visualization Demo

This notebook demonstrates a simplified channel-level visualization workflow for the Python-based fNIRS MA analysis pipeline.

The goal is to visualize which fNIRS channels show significant MA-related activation effects.

Important note: this notebook visualizes channel-level results only. It does not interpret channels as specific brain regions.


## Why channel-level visualization first?

The original MATLAB pipeline visualizes significant channels on a 3D brain template. However, anatomical interpretation requires reliable channel coordinates and channel-to-region mapping.

For the current BrainHack project, we first reproduce a conservative channel-level visualization:

- Which channels are significant?
- Are the effects positive or negative?
- Are the results uncorrected or FWE-corrected?

Brain-region interpretation can be added later after confirming the coordinate file and anatomical mapping.


In [ ]:
from pathlib import Path
import importlib.util

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate project root
cwd = Path.cwd()
project_root = cwd if (cwd / 'src').exists() else cwd.parent

# Load local src/statistics.py safely
stats_path = project_root / 'src' / 'statistics.py'
stats_spec = importlib.util.spec_from_file_location('fnirs_statistics', stats_path)
fnirs_statistics = importlib.util.module_from_spec(stats_spec)
stats_spec.loader.exec_module(fnirs_statistics)

# Load local src/visualization.py safely
viz_path = project_root / 'src' / 'visualization.py'
viz_spec = importlib.util.spec_from_file_location('fnirs_visualization', viz_path)
fnirs_visualization = importlib.util.module_from_spec(viz_spec)
viz_spec.loader.exec_module(fnirs_visualization)

add_significance_flags = fnirs_statistics.add_significance_flags
get_fwe_threshold = fnirs_statistics.get_fwe_threshold

create_demo_channel_layout = fnirs_visualization.create_demo_channel_layout
plot_channel_effects = fnirs_visualization.plot_channel_effects
plot_significant_channels = fnirs_visualization.plot_significant_channels


## Step 1: Create simulated channel-level statistical results

In the final pipeline, this table should come from the group-level GLM output.

Here, simulated results are used only to test the visualization workflow.


In [ ]:
np.random.seed(42)

channels = [f'Ch{i:02d}' for i in range(1, 33)]

results_df = pd.DataFrame({
    'channel': channels,
    'contrast': 'Group_difference_MA_minus_Control',
    'effect': np.random.normal(loc=0, scale=0.25, size=32),
    'p_value': np.random.uniform(0, 0.10, size=32)
})

# Manually create a few significant channels for demonstration
results_df.loc[0, ['effect', 'p_value']] = [0.55, 0.0008]
results_df.loc[10, ['effect', 'p_value']] = [-0.48, 0.0010]
results_df.loc[20, ['effect', 'p_value']] = [0.32, 0.0300]

results_df.head()


## Step 2: Add significance flags

The MATLAB pipeline reports both uncorrected and FWE-corrected results.

The FWE threshold is:

`p < .05 / 32`


In [ ]:
fwe_threshold = get_fwe_threshold(alpha=0.05, n_tests=32)
fwe_threshold


In [ ]:
results_with_flags = add_significance_flags(
    results_df,
    p_col='p_value',
    effect_col='effect',
    alpha=0.05,
    n_tests=32
)

results_with_flags.head()


## Step 3: Create a simple channel layout

This layout is only a simplified demonstration layout. It is not a real anatomical layout.


In [ ]:
layout_df = create_demo_channel_layout(n_channels=32, n_cols=8)
layout_df.head()


## Step 4: Plot channel-level effect values

This plot shows the effect value for each channel.


In [ ]:
fig = plot_channel_effects(
    results_with_flags,
    layout_df,
    effect_col='effect',
    p_col='p_value',
    title='Demo channel-level effects',
    save_path=project_root / 'figures' / 'channel_plots' / '05_demo_channel_effects.png'
)

plt.show()


## Step 5: Plot FWE-significant channels

This plot highlights channels that survive FWE correction.

Positive and negative significant effects are marked separately.


In [ ]:
fig = plot_significant_channels(
    results_with_flags,
    layout_df,
    significance_col='significant_fwe',
    direction_col='direction',
    title='Demo FWE-significant channels',
    save_path=project_root / 'figures' / 'channel_plots' / '05_demo_significant_channels.png'
)

plt.show()


## Step 6: List significant channels

The table below lists the FWE-significant channels.


In [ ]:
fwe_significant_channels = results_with_flags[
    results_with_flags['significant_fwe']
].copy()

fwe_significant_channels[['channel', 'contrast', 'effect', 'p_value', 'direction']]


## Summary

This notebook demonstrates how to visualize significant fNIRS channels at the channel level.

At the current stage, the results should be interpreted as significant channels, not as specific brain regions.

The next step is to connect this visualization workflow to real or anonymized group-level GLM results.
